# Module 3: Covariance Estimation & Comparison

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

The covariance matrix is the **single most important input** to portfolio optimization.
A noisy covariance estimate → garbage portfolios. In this module we:

1. Compare 7 covariance estimation methods
2. Analyze eigenvalue spectra vs Random Matrix Theory (Marchenko-Pastur)
3. Evaluate which estimators produce the most stable Minimum Variance Portfolios
4. Test out-of-sample stability through rolling window analysis
5. Select the best estimator(s) for use in portfolio optimization (Module 4)

### Methods Compared

| Method | Description | Pros | Cons |
|--------|-------------|------|------|
| **Sample** | Standard MLE | Unbiased | Very noisy when n/T is high |
| **Ledoit-Wolf** | Shrinkage toward identity | Analytically optimal | Assumes specific structure |
| **OAS** | Oracle Approx Shrinkage | Better for large p | Similar assumptions |
| **EWMA** | Exponential weighting | Captures regime changes | Requires halflife tuning |
| **Constant Corr** | Average correlation | Very stable | Too restrictive |
| **Denoised (RMT)** | Random Matrix Theory | Removes noise eigenvalues | Assumes noise/signal split |
| **Gerber** | Concordance statistic | Robust to noise | Newer, less validated |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
import sys
import os
import json

sys.path.insert(0, os.path.abspath('..'))

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')

print('Setup complete.')

In [ ]:
# Load processed data from Module 1
data_dir = '../data/processed'

daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

# Use only investable assets
signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

print(f'Investable assets: {len(investable)}')
print(f'Observations: {len(returns)}')
print(f'Ratio (n/T): {len(investable)/len(returns):.4f}')
print(f'Date range: {returns.index[0].date()} to {returns.index[-1].date()}')

## 1. Compute All Covariance Estimates

In [ ]:
from project.covariance import CovarianceEstimator, CovarianceEvaluator

estimator = CovarianceEstimator(returns)
all_cov = estimator.estimate_all(ewma_halflife=63)

print(f'Computed {len(all_cov)} covariance estimators:')
for name in all_cov:
    print(f'  ✓ {name}')

## 2. Eigenvalue Analysis

**Why eigenvalues matter:**
- Large condition number → matrix is near-singular → MVP weights blow up
- Near-zero eigenvalues = estimated "phantom" risk directions from noise
- Effective rank tells us how many independent risk factors truly exist

**Random Matrix Theory (Marchenko-Pastur)**: for a purely random correlation matrix
of dimension $n × T$, eigenvalues fall within $[(1-\sqrt{n/T})^2, (1+\sqrt{n/T})^2]$.
Eigenvalues above this bound are "signal"; below are "noise".

In [ ]:
evaluator = CovarianceEvaluator(returns)

eigen_analysis = evaluator.eigenvalue_analysis(all_cov)
print('Eigenvalue Analysis:')
print('=' * 100)
print(eigen_analysis.to_string())

In [ ]:
fig = evaluator.plot_eigenvalue_spectrum(all_cov)
plt.show()

## 3. Minimum Variance Portfolio Comparison

The acid test: **how different are the MVP weights under each estimator?**

If the sample covariance produces extreme weights (large short positions,
high gross exposure), it confirms the matrix is too noisy for optimization.
Shrinkage and denoising should produce more reasonable portfolios.

In [ ]:
mvp_summary, mvp_portfolios = evaluator.min_variance_portfolios(all_cov)

print('Minimum Variance Portfolio Summary:')
print('=' * 80)
print(mvp_summary.round(2).to_string())

In [ ]:
fig = evaluator.plot_mvp_comparison(mvp_portfolios)
plt.show()

## 4. Correlation Differences Between Estimators

Visualize how much the estimated correlation matrices differ.
Large differences → the estimator choice materially affects your portfolio.

In [ ]:
fig = evaluator.plot_correlation_differences(all_cov, reference='Sample')
plt.show()

In [ ]:
# Frobenius distance matrix
dist = evaluator.pairwise_distances(all_cov, metric='correlation')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(dist, annot=True, fmt='.2f', cmap='YlOrRd', square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Frobenius Distance Between Correlation Estimators',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Denoised Covariance Deep Dive

The denoised estimator uses Random Matrix Theory to separate
signal eigenvalues from noise. Let's see how many eigenvalues
are classified as signal vs noise.

In [ ]:
denoised = all_cov.get('Denoised (RMT)', {})
if denoised:
    print(f"Signal eigenvalues: {denoised.get('n_signal', 'N/A')}")
    print(f"Noise eigenvalues: {denoised.get('n_noise', 'N/A')}")
    print(f"Marchenko-Pastur upper bound: {denoised.get('mp_upper_bound', 'N/A'):.3f}")

    eig_orig = denoised.get('eigenvalues_original', np.array([]))
    eig_clean = denoised.get('eigenvalues_denoised', np.array([]))

    if len(eig_orig) > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        x = range(1, len(eig_orig) + 1)
        ax.bar(x, eig_orig, alpha=0.5, label='Original', color='steelblue')
        ax.bar(x, eig_clean, alpha=0.5, label='Denoised', color='orange')
        ax.axhline(y=denoised.get('mp_upper_bound', 0), color='red', linestyle='--',
                   label=f"MP bound ({denoised.get('mp_upper_bound', 0):.2f})")
        ax.set_xlabel('Component')
        ax.set_ylabel('Eigenvalue')
        ax.set_title('Eigenvalues: Original vs Denoised (RMT)', fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 6. Out-of-Sample Stability

The ultimate test: **rolling window MVP analysis**.

A good estimator should produce:
- Stable volatility estimates over time
- Low portfolio turnover (weights don't flip every month)
- Reasonable weight bounds (no extreme leverage)

In [ ]:
fig = evaluator.plot_stability_comparison(
    methods=['sample', 'ledoit_wolf', 'denoised', 'gerber'],
    window=252,
    step=21
)
plt.show()

## 7. Shrinkage Intensity Analysis

How much does Ledoit-Wolf shrink the sample covariance?
Higher shrinkage = more regularization needed = noisier sample.

In [ ]:
# Shrinkage intensity comparison
lw_result = all_cov.get('Ledoit-Wolf', {})
oas_result = all_cov.get('OAS', {})

print('Shrinkage Intensities:')
print(f"  Ledoit-Wolf: {lw_result.get('shrinkage', 'N/A'):.4f}")
print(f"  OAS:         {oas_result.get('shrinkage', 'N/A'):.4f}")
print(f"\nInterpretation:")
print(f"  0 = no shrinkage (trust sample fully)")
print(f"  1 = maximum shrinkage (ignore sample, use target only)")

## 8. Recommendation & Export

Based on the analysis, select the best estimator(s) for Module 4.

In [ ]:
# Export selected covariance matrices for Module 4
import pickle

export = {
    'Ledoit-Wolf': all_cov['Ledoit-Wolf'],
    'Denoised (RMT)': all_cov.get('Denoised (RMT)', all_cov.get('Sample')),
    'Gerber': all_cov.get('Gerber', all_cov.get('Sample')),
    'Sample': all_cov['Sample'],
}

export_path = '../data/processed/covariance_estimates.pkl'
with open(export_path, 'wb') as f:
    pickle.dump(export, f)
print(f'Exported {len(export)} covariance estimates to {export_path}')

# Also export just the Ledoit-Wolf as the default
lw_cov = all_cov['Ledoit-Wolf']['covariance']
lw_cov.to_parquet('../data/processed/covariance_lw.parquet')
print('Default covariance (Ledoit-Wolf) saved.')

---

## Key Takeaways from Module 3

1. **Sample covariance is dangerously noisy** — high condition number, extreme MVP weights, unstable over time
2. **Ledoit-Wolf** is a reliable default — significant shrinkage reduces noise, stable portfolios
3. **Denoised (RMT)** removes noise eigenvalues, revealing the true correlation structure
4. **Gerber statistic** provides a robust alternative that ignores noise-level moves
5. **Eigenvalue analysis** shows most eigenvalues are noise (Marchenko-Pastur), confirming the need for regularization
6. **The estimator choice matters enormously** — different estimators produce very different MVPs

### For Module 4 (Portfolio Optimization)

We will use **Ledoit-Wolf as default** and compare results with Denoised and Gerber.
The sample covariance will serve as a cautionary baseline.

### Next: Module 4 — Portfolio Optimization

Markowitz Mean-Variance, HRP, Risk Parity, inverse volatility, and Mean-CVaR optimization. Black-Litterman is excluded from production unless dated, sourced investor views are supplied.